In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)
import torch
from PIL import Image
import torchvision
from sklearn.model_selection import train_test_split
from matplotlib import pyplot as plt

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
dirname = '/kaggle/input/covid19-radiography-database/COVID-19_Radiography_Dataset/'
for filename in os.listdir(dirname):
    print(os.path.join(dirname, filename))

torch.manual_seed(0)
# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [ ]:
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
print(device)

# Create Dataset directory and split

In [ ]:
def create_image_dataset(dirname=dirname,
                         sources=['Normal','Lung_Opacity','Viral Pneumonia','COVID'],
                         labels=['normal','lung_opacity','viral_pneumonia','covid']):
    '''
    Create a master dataset of all image paths and labels
    '''
    image_dataset = pd.DataFrame(columns=['image', 'label'])
    for source, label in zip(sources, labels):
        files = os.listdir(os.path.join(dirname, source))

        temp_df = pd.DataFrame({
             'image': [os.path.join(dirname, source, item) for item in files if item.lower().endswith('.png')]
         })
        temp_df['label'] = label

        image_dataset = pd.concat([image_dataset, temp_df], ignore_index=True)
    return image_dataset

In [ ]:
image_dataset = create_image_dataset()
train_dataset_paths, test_dataset_paths = train_test_split(image_dataset, test_size=0.15, stratify=image_dataset['label'])

# Dataset

In [ ]:
class XRayDataset(torch.utils.data.Dataset):
    def __init__(self, image_dataset, transform):
        self.image_dataset = image_dataset
            
        self.class_names = self.image_dataset['label'].unique().tolist()
        self.transform = transform
        
        print(self.image_dataset['label'].value_counts())
        
    def __len__(self):
        return len(self.image_dataset)
    
    def __getitem__(self, index):
        row = self.image_dataset.iloc[index]
        image_path = row['image']
        label = row['label']
        
        image = Image.open(image_path).convert('RGB')
        
        return self.transform(image), self.class_names.index(label)

# Transformers

In [ ]:
train_transform = torchvision.transforms.Compose([
    torchvision.transforms.Resize(size=(224, 224)),
    torchvision.transforms.RandomHorizontalFlip(),
    torchvision.transforms.ToTensor(),
    torchvision.transforms.Normalize(mean=[0.485, 0.456, 0.406],
                                    std=[0.229, 0.224, 0.225])
])

test_transform = torchvision.transforms.Compose([
    torchvision.transforms.Resize(size=(224, 224)),
    torchvision.transforms.ToTensor(),
    torchvision.transforms.Normalize(mean=[0.485, 0.456, 0.406],
                                    std=[0.229, 0.224, 0.225])
])

# Dataloader

In [ ]:
print('Train dataset')
train_dataset = XRayDataset(train_dataset_paths, transform=train_transform)
print('\nTest dataset')
test_dataset = XRayDataset(test_dataset_paths, transform=test_transform)

dl_train = torch.utils.data.DataLoader(train_dataset, batch_size=1, shuffle=True)
dl_test = torch.utils.data.DataLoader(test_dataset, batch_size=6, shuffle=True)

print('Num of training batches', len(dl_train))
print('Num of test batches', len(dl_test))

In [ ]:
class_names = train_dataset.class_names

def show_images(images, labels, preds):
    plt.figure(figsize=(8,4))
    for i, image in enumerate(images):
        plt.subplot(1,6, i+1, xticks=[], yticks=[])
        image = image.numpy().transpose((1,2,0))
        mean = np.array([0.485, 0.456, 0.406])
        std = np.array([0.229, 0.224, 0.225])
        image = image*std+mean
        image = np.clip(image, 0., 1.)
        plt.imshow(image)
        
        col = 'green' if preds[i]==labels[i] else 'red'
        
        plt.xlabel(f'{class_names[int(labels[i].numpy())]}')
        plt.ylabel(f'{class_names[int(preds[i].numpy())]}', color=col)
    plt.tight_layout()
    plt.show()

In [ ]:
num_class = len(train_dataset.class_names)
resnet18 = torchvision.models.resnet18(pretrained=True)
resnet18.fc = torch.nn.Linear(in_features=512, out_features=num_class, bias=True)
resnet18 = resnet18.to(device)
print(resnet18)

In [ ]:
loss_fn = torch.nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(resnet18.parameters(), lr=3e-5)

In [ ]:
def show_preds():
    resnet18.eval()
    images, labels = next(iter(dl_test))
    outputs = resnet18(images)
    _, preds = torch.max(outputs, 1)
    show_images(images.cpu(), labels.cpu(), preds.cpu())

# Training model

In [ ]:
def train(epochs):
    print('Training started...')
    for e in range(epochs):
        print('='*20)
        print(f'Starting epochs {e+1}/{epochs}')
        print('='*20)
        
        train_loss = 0
        resnet18.train()
        for train_step, (images, labels) in enumerate(dl_train):
            optimizer.zero_grad()
            images = images.to(device, dtype=torch.float)
            labels = labels.to(device, dtype=torch.long)
            outputs = resnet18(images)
            loss = loss_fn(outputs, labels)
            loss.backward()
            optimizer.step()
            train_loss += loss
            
            if train_step%20 == 0:
                print(f'Evaluating at step {train_step}')
                acc = 0
                val_loss = 0
                resnet18.eval()
                
                for val_step, (images, labels) in enumerate(dl_test):
                    images = images.to(device, dtype=torch.float)
                    labels = labels.to(device, dtype=torch.long)
                    outputs = resnet18(images)
                    loss = loss_fn(outputs, labels)
                    val_loss += loss
                    
                    _, preds = torch.max(outputs, 1)
                    acc += sum((labels == preds).cpu().numpy())
                acc /= val_step
                val_loss/=(val_step + 1)
                print(f'Validation Loss: {val_loss:.4f}, Accuracy: {acc:.4f}')
                show_preds()
                
                resnet18.train()
                
                if acc>0.95:
                    print('Performance condition met...')
                    return 
        train_loss = train_loss/(train_step + 1)
        print(f'Training loss: {train_loss:.4f}')

In [ ]:
train(1)

# Final Result

In [ ]:
show_preds()